# NeuroSeg-X: Brain MRI Multi-Task Framework
### VERSION: 2.3 (STABLE ASCII + BRUTE-FORCE SCANNER)

**Fixes in this version:**
1. **Super Scanner**: Uses fuzzy matching and full-tree search to pair images and masks.
2. **Diagnostics**: Added a cell to list the actual folder structure if data is missing.
3. **Safe Dataloader**: Added explicit checks to prevent the 'num_samples=0' crash.
4. **Pure ASCII**: No emojis or special characters.

In [ ]:
# -------------------------------------------------------------------
# STEP 1: INSTALLS & IMPORTS
# -------------------------------------------------------------------
!pip install -q albumentations opencv-python-headless tensorboard tqdm torchmetrics scikit-learn

import os
import random
import shutil
import zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm 
from PIL import Image
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from google.colab import drive, files

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, jaccard_score

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# -------------------------------------------------------------------
# STEP 2: CONFIGURATION
# -------------------------------------------------------------------
config = {
    'image_size': [512, 512],
    'batch_size': 4,
    'num_epochs': 50,
    'lr': 1e-4,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'save_dir': '/content/results',
    'data_dir': '/content/data',
    'seed': 42
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(config['seed'])
os.makedirs(config['save_dir'], exist_ok=True)
print("[OK] Configuration saved.")

In [ ]:
# -------------------------------------------------------------------
# STEP 3: DATA EXTRACTION
# -------------------------------------------------------------------
try:
    drive.mount('/content/drive')
except:
    print("[INFO] Drive already mounted or error.")

possible_paths = [
    '/content/drive/MyDrive/Colab_Notebooks_Data',
    '/content/drive/MyDrive/Colab_Notebook_Data',
    '/content/drive/MyDrive/Data'
]

drive_path = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])
datasets = ['brain_glioma.zip', 'brain_menin.zip', 'brain_tumor.zip']
raw_extract = '/content/raw_data'
os.makedirs(raw_extract, exist_ok=True)

print(f"[INFO] Searching for ZIPs in: {drive_path}")
found_any = False
for ds in datasets:
    zp = os.path.join(drive_path, ds)
    if os.path.exists(zp):
        print(f"  - Extracting {ds}...")
        with zipfile.ZipFile(zp, 'r') as z: z.extractall(raw_extract)
        found_any = True
    else:
        print(f"  [WARN] {ds} NOT found in Drive.")

if not found_any:
    print("!!! WARNING: No zip files were found. Scanning raw_data for existing files...")

print("[OK] Step 3 complete.")

In [ ]:
# -------------------------------------------------------------------
# STEP 4: BRUTE-FORCE DATA PAIRING
# -------------------------------------------------------------------
print("[INFO] Starting Brute-Force Pairing...")

data_root = Path(config['data_dir'])
for split in ['train', 'val']: 
    (data_root / split / 'images').mkdir(parents=True, exist_ok=True)
    (data_root / split / 'masks').mkdir(parents=True, exist_ok=True)

raw_p = Path('/content/raw_data')
all_files = sorted(list(raw_p.rglob('*')))
exts = ['.png', '.jpg', '.jpeg', '.tif', '.nii']

# 1. Categorize all images and potential masks
total_images = []
total_masks = {}

print("  Scanning file tree...")
for f in all_files:
    if not f.is_file(): continue
    if f.name.startswith('.'): continue
    if f.suffix.lower() not in exts: continue
    
    name_low = f.name.lower()
    is_mask = any(x in name_low for x in ['mask', 'seg', 'label', 'gt'])
    
    if is_mask:
        # Use stem as key for matching (e.g. 'case123_mask' -> 'case123')
        key = f.stem.replace('_mask', '').replace('_seg', '').replace('_Segmentation', '')
        total_masks[key.lower()] = f
    else:
        total_images.append(f)

print(f"  [DEBUG] Found {len(total_images)} images and {len(total_masks)} potential masks.")

# 2. Brute-force match images to masks
all_samples = []
for img_p in tqdm(total_images, desc="Pairing", ascii=True):
    key = img_p.stem.lower()
    mask_p = total_masks.get(key)
    
    # Fallback to name search if key matching fails
    if not mask_p:
        for mk in total_masks.keys():
            if mk == key or mk in key or key in mk:
                mask_p = total_masks[mk]
                break
    
    if mask_p:
        path_str = str(img_p).lower()
        label_grading = 1 if 'glioma' in path_str else 0
        all_samples.append({'img': img_p, 'mask': mask_p, 'grading': label_grading})

print(f"\n[RESULTS] Paired {len(all_samples)} samples.")

if len(all_samples) == 0:
    print("!!! ERROR: NO SAMPLES PAIRED. Here is the structure of your extraction folder:")
    for root, dirs, files in os.walk('/content/raw_data'):
        level = root.replace('/content/raw_data', '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files[:5]: print('{}{}'.format(subindent, f))
else:
    # Split and Copy
    random.shuffle(all_samples)
    split_idx = int(0.8 * len(all_samples))
    splits = {'train': all_samples[:split_idx], 'val': all_samples[split_idx:]}

    for s_name, s_data in splits.items():
        print(f"  Copying {s_name} set ({len(s_data)} samples)...")
        for sample in tqdm(s_data, ascii=True):
            unique_name = f"g{sample['grading']}_{sample['img'].stem}{sample['img'].suffix}"
            shutil.copy2(sample['img'], data_root / s_name / 'images' / unique_name)
            shutil.copy2(sample['mask'], data_root / s_name / 'masks' / unique_name)

    print("[OK] Data organization complete.")

In [ ]:
# -------------------------------------------------------------------
# STEP 5: MODELS & DATASET
# -------------------------------------------------------------------
class NeuroSegDataset(Dataset):
    def __init__(self, split, transforms=None):
        self.img_dir = Path(config['data_dir']) / split / 'images'
        self.mask_dir = Path(config['data_dir']) / split / 'masks'
        self.files = sorted(list(self.img_dir.glob('*')))
        self.transforms = transforms
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img_path = self.files[idx]
        mask_path = self.mask_dir / img_path.name
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        if mask.max() > 1: mask /= 255.0
        grading = int(img_path.name[1])
        if self.transforms: 
            aug = self.transforms(image=image, mask=mask)
            image, mask = aug['image'], aug['mask']
        return {'image': image, 'mask': mask.unsqueeze(0), 'detection': torch.tensor(1.0), 'grading': torch.tensor(grading)}

def get_tf():
    tr = A.Compose([A.Resize(512, 512), A.HorizontalFlip(), A.Normalize(), ToTensorV2()])
    vl = A.Compose([A.Resize(512, 512), A.Normalize(), ToTensorV2()])
    return tr, vl

class DoubleConv(nn.Module):
    def __init__(self, i, o): 
        super().__init__()
        self.c = nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(), nn.Conv2d(o, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU())
    def forward(self, x): return self.c(x)

class NeuroSegX(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = DoubleConv(3, 64)
        self.e2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.seg = nn.Conv2d(128, 1, 1)
        self.det = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 1), nn.Sigmoid())
        self.grd = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 2))
    def forward(self, x):
        x1 = self.e1(x)
        x2 = self.e2(x1)
        return {'segmentation': self.seg(x2), 'detection': self.det(x2), 'grading': self.grd(x2)}

print("[OK] Setup complete.")

In [ ]:
# -------------------------------------------------------------------
# STEP 6: TRAINING LOOP
# -------------------------------------------------------------------
train_path = config['data_dir'] + '/train/images'
if not os.path.exists(train_path) or len(os.listdir(train_path)) == 0:
    print("!!! CRITICAL: Train folder is EMPTY. Check Step 4 output for errors. !!!")
else:
    tr_tf, vl_tf = get_tf()
    dataset_tr = NeuroSegDataset('train', tr_tf)
    tr_loader = DataLoader(dataset_tr, batch_size=config['batch_size'], shuffle=True)
    vl_loader = DataLoader(NeuroSegDataset('val', vl_tf), batch_size=config['batch_size'])

    model = NeuroSegX().to(config['device'])
    opt = optim.AdamW(model.parameters(), config['lr'])
    scaler = GradScaler()
    c_seg, c_det, c_grd = nn.BCEWithLogitsLoss(), nn.BCELoss(), nn.CrossEntropyLoss()

    print(f"[INFO] Training {len(dataset_tr)} samples for {config['num_epochs']} epochs...")
    for epoch in range(config['num_epochs']):
        model.train()
        tl = 0
        for b in tqdm(tr_loader, desc=f"Epoch {epoch+1}", ascii=True):
            imgs, masks = b['image'].to(config['device']), b['mask'].to(config['device'])
            det_gt, grd_gt = b['detection'].to(config['device']), b['grading'].to(config['device'])
            with autocast():
                out = model(imgs)
                mask_out = F.interpolate(out['segmentation'], size=(512,512), mode='bilinear')
                loss = 0.5*c_seg(mask_out, masks) + 0.25*c_det(out['detection'], det_gt.unsqueeze(1)) + 0.25*c_grd(out['grading'], grd_gt)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tl += loss.item()
        print(f"[INFO] Epoch {epoch+1} Completed. Loss: {tl/len(tr_loader):.4f}")
    
    torch.save(model.state_dict(), '/content/neuroseg_final.pth')
    print("[OK] Model saved.")

In [ ]:
# -------------------------------------------------------------------
# STEP 7: EVALUATION & VISUALIZATION
# -------------------------------------------------------------------
if 'vl_loader' in globals():
    print("[INFO] Evaluating...")
    model.eval()
    dices = []
    with torch.no_grad():
        for batch in tqdm(vl_loader, ascii=True):
            imgs, masks = batch['image'].to(config['device']), batch['mask'].to(config['device'])
            out = model(imgs)
            pred_mask = (torch.sigmoid(F.interpolate(out['segmentation'], (512,512))) > 0.5).float()
            for i in range(imgs.size(0)):
                p, g = pred_mask[i].view(-1), masks[i].view(-1)
                inter = (p * g).sum()
                dices.append(((2.*inter)/(p.sum()+g.sum()+1e-8)).item())
    print(f"[RESULTS] Mean Dice Score: {np.mean(dices):.4f}")

    test_batch = next(iter(vl_loader))
    with torch.no_grad():
        out = model(test_batch['image'].to(config['device']))
        pred = (torch.sigmoid(F.interpolate(out['segmentation'], (512,512))) > 0.5).cpu()

    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1); plt.imshow(test_batch['image'][0].permute(1,2,0)); plt.title("Original")
    plt.subplot(1,2,2); plt.imshow(pred[0,0], cmap='jet'); plt.title("Prediction")
    plt.show()
    files.download('/content/neuroseg_final.pth')
else:
    print("!!! Evaluation skipped (No validation data). !!!")